# 07 · Targeting policy and business impact

**Goal:** turn the findings into a decision: for each customer, send the Mens email, the Womens email, or nothing. Then estimate what each policy would deliver, compared with (a) emailing everyone and (b) emailing no one, on held-out customers, with uncertainty and stated assumptions.

**Objective:** incremental spend per customer over the two-week window, **net of the cost of sending the email**.

### Assumptions (read these before the numbers)

1. **Two-week horizon.** Effects after two weeks (purchases pulled forward or delayed, unsubscribes, email fatigue) are not in the data and are ignored.
2. **Revenue, not profit.** `spend` is revenue. With a gross margin $m$, every dollar figure below should be multiplied by $m$ before comparing it with costs; the break-even costs shown here are therefore upper bounds.
3. **Email cost is unknown.** Instead of assuming one number, results are shown across a range of costs per email, from \$0 to \$1.
4. **Visit uplift is used as the targeting signal.** Notebook 06 showed that no spend model beats random targeting, while visit models do. Visit uplift is converted to dollars with one number per email: the average extra spend per extra visit, estimated on the training set. This assumes an extra visit is worth the same, on average, for every customer who receives a given email, a simplification.
5. **The data is from 2008.** Customer behaviour and email performance today may differ.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from hillstrom.data import load_customers
from hillstrom.plots import ARM_COLORS, ARM_LABELS, TEXT_PRIMARY, TEXT_SECONDARY, apply_style, save_figure
from hillstrom.policy import assign_policy, bootstrap_policy_value, policy_value_ipw
from hillstrom.uplift import TLearner, make_features, split_train_test

apply_style()
df = load_customers()
train, test = split_train_test(df)  # same split and seed as notebook 06
EMAILS = ["mens", "womens"]
arm_test = test["arm"].to_numpy()
n_test = len(test)
print(f"Policies are learned on {len(train):,} training customers and evaluated on {n_test:,} test customers.")

Policies are learned on 44,800 training customers and evaluated on 19,200 test customers.


## 1. From visit uplift to dollars

For each email, the training set gives the average effect on visits and on spend. Their ratio is the average extra spend per extra visit.

In [2]:
dollars_per_visit = {}
rows = []
for email in EMAILS:
    tr = train[train["arm"].isin([email, "control"])]
    is_email = tr["arm"] == email
    visit_effect = tr.loc[is_email, "visit"].mean() - tr.loc[~is_email, "visit"].mean()
    spend_effect = tr.loc[is_email, "spend"].mean() - tr.loc[~is_email, "spend"].mean()
    dollars_per_visit[email] = spend_effect / visit_effect
    rows.append({"email": ARM_LABELS[email], "visit_effect_pp": 100 * visit_effect,
                 "spend_effect": spend_effect, "dollars_per_extra_visit": dollars_per_visit[email]})
pd.DataFrame(rows).style.format(
    {"visit_effect_pp": "{:+.2f}", "spend_effect": "${:+.3f}", "dollars_per_extra_visit": "${:.2f}"}
).hide(axis="index")

email,visit_effect_pp,spend_effect,dollars_per_extra_visit
Mens E-Mail,+7.78,$+0.771,$9.92
Womens E-Mail,+4.68,$+0.472,$10.08


Both emails turn an extra visit into about \$10 of extra spend on average. Each customer's estimated dollar uplift is their predicted visit uplift times this number.

The two targeting signals from notebook 06 are refit here on the training set:

- **Segment rule:** each customer gets their `purchase_group`'s visit effect (three numbers per email).
- **Linear T-learner:** each customer gets their individual predicted visit uplift.

In [3]:
X_test = make_features(test)
dollar_uplift = {}
for email in EMAILS:
    tr = train[train["arm"].isin([email, "control"])]
    treated = (tr["arm"] == email).to_numpy()

    model = TLearner("binary", base="linear").fit(make_features(tr), tr["visit"].to_numpy(), treated)
    dollar_uplift["Linear T-learner", email] = model.predict_uplift(X_test) * dollars_per_visit[email]

    group_effect = {
        group: g.loc[g["arm"] == email, "visit"].mean() - g.loc[g["arm"] == "control", "visit"].mean()
        for group, g in tr.groupby("purchase_group")
    }
    dollar_uplift["Segment rule", email] = test["purchase_group"].map(group_effect).to_numpy() * dollars_per_visit[email]

TARGETED = ["Segment rule", "Linear T-learner"]
FIXED = {"No email": "control", "Mens E-Mail to all": "mens", "Womens E-Mail to all": "womens"}


def policy(name: str, cost: float) -> np.ndarray:
    # The action for every test customer under a named policy at a given cost per email.
    if name in FIXED:
        return np.full(n_test, FIXED[name])
    return assign_policy(dollar_uplift[name, "mens"], dollar_uplift[name, "womens"], cost)

**Decision rule for the targeted policies:** for each customer, pick the action with the largest estimated net gain: Mens email (dollar uplift − cost), Womens email (dollar uplift − cost), or nothing (0). A customer is emailed only if the best net gain is positive.

## 2. How a policy is evaluated: inverse propensity weighting (IPW)

A policy was never actually run, so its value has to be estimated from the experiment. In the experiment every customer had a 1/3 chance of receiving each action. The customers whose *actual* arm happens to match what the policy would choose are therefore a random third of the customers the policy would treat that way. Their outcomes, weighted by 1/(1/3) = 3, estimate what the policy would deliver:

$$\hat V(\pi) = \frac{1}{n}\sum_{i=1}^{n} \frac{\mathbb{1}[A_i = \pi(x_i)]\; Y_i}{1/3} \;-\; \text{cost} \times \text{share emailed}.$$

This uses only observed outcomes, not the model's own predictions, so a wrong model cannot make its own policy look good. The price is noise: each policy's value rests on about a third of the test customers. Intervals come from 1,000 bootstrap resamples of the test customers, using the same resamples for every policy so that policies can be compared pairwise.